# LeetCode 3633 — Earliest Finish Time for Land and Water Rides I bằng Python

Link đề: https://leetcode.com/problems/earliest-finish-time-for-land-and-water-rides-i/ · Độ khó: **Easy** · Chủ đề: Array, Two Pointers, Greedy, Sorting, Binary Search

Skeleton LeetCode:

```python
class Solution:
    def earliestFinishTime(self, landStartTime: List[int], landDuration: List[int],
                           waterStartTime: List[int], waterDuration: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chọn **đúng một** trò chơi trên cạn và **đúng một** trò chơi dưới nước, chơi theo **một thứ tự bất kỳ**, mỗi trò chỉ bắt đầu được từ thời điểm mở cửa trở đi; tìm **thời điểm sớm nhất** chơi xong cả hai.
2. Mô hình hóa việc "chơi trò thứ hai sau khi xong trò thứ nhất": `bắt đầu = max(thời điểm vừa xong, giờ mở cửa)`, `kết thúc = bắt đầu + thời lượng`.
3. Thấy rằng với **thứ tự cố định** (cạn rồi nước), kết quả tối ưu chọn trò thứ nhất có **thời điểm kết thúc nhỏ nhất** (tham lam), rồi chọn trò thứ hai tốt nhất.
4. Viết 4 cách: duyệt mọi cặp và mọi thứ tự, tham lam `O(n + m)`, tìm kiếm nhị phân theo đáp án, và tính sẵn thời điểm kết thúc rồi duyệt; đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh các lỗi: quên một thứ tự, quên chờ giờ mở cửa, chọn trò **mở sớm nhất** thay vì **kết thúc sớm nhất**, và chọn trò **ngắn nhất**.

## 2. Đề bài & ví dụ

Công viên có `n` trò chơi trên cạn và `m` trò chơi dưới nước.

- Trò trên cạn `i` mở cửa từ `landStartTime[i]` và kéo dài `landDuration[i]`.
- Trò dưới nước `j` mở cửa từ `waterStartTime[j]` và kéo dài `waterDuration[j]`.

Du khách phải chơi **đúng một trò mỗi loại**, theo **thứ tự bất kỳ** (cạn trước hoặc nước trước). Một trò có thể bắt đầu **đúng lúc mở cửa hoặc muộn hơn**. Nếu bắt đầu ở thời điểm `t` thì kết thúc ở `t + duration`. Chơi xong một trò, du khách có thể vào trò kia **ngay** khi nó mở (nếu chưa mở thì chờ). Trả về **thời điểm sớm nhất** du khách chơi xong cả hai trò.

| Ví dụ | input | Output | Giải thích |
|---|---|---|---|
| 1 | `landStartTime=[2,8]`, `landDuration=[4,1]`, `waterStartTime=[6]`, `waterDuration=[3]` | `9` | cạn 0 (2 -> 6), rồi nước 0 (mở 6, 6 -> 9) |
| 2 | `landStartTime=[5]`, `landDuration=[3]`, `waterStartTime=[1]`, `waterDuration=[10]` | `14` | nước 0 (1 -> 11), rồi cạn 0 (đã mở, 11 -> 14); thứ tự ngược lại ra `18` |

Constraints: `1 <= n, m <= 100`; `landStartTime.length == landDuration.length == n`; `waterStartTime.length == waterDuration.length == m`; mọi giá trị là số nguyên trong `[1, 1000]`.

In [1]:
landStartTime, landDuration = [2, 8], [4, 1]
waterStartTime, waterDuration = [6], [3]

# cạn 0 rồi nước 0
f1 = landStartTime[0] + landDuration[0]
s2 = max(f1, waterStartTime[0])
print("cạn xong lúc", f1, "-> nước bắt đầu", s2, "-> xong", s2 + waterDuration[0])

cạn xong lúc 6 -> nước bắt đầu 6 -> xong 9


## 3. Kiến thức nền

### 3.1. Mô phỏng "xong trò này rồi vào trò kia"

Nếu vừa xong ở thời điểm `done` và trò kế tiếp mở cửa lúc `start` thì có hai tình huống: đến sớm (`done < start`) phải chờ đến `start`; đến muộn (`done >= start`) vào ngay. Cả hai gộp thành `begin = max(done, start)`. Kết thúc `begin + duration`.

In [2]:
def play(done, start, dur):
    begin = max(done, start)
    return begin + dur

print(play(6, 6, 3))     # vào đúng lúc mở
print(play(2, 10, 3))    # đến sớm: chờ đến 10
print(play(11, 5, 3))    # đến muộn: vào ngay

9
13
14


### 3.2. `zip`, `min` và biểu thức sinh

`zip(landStartTime, landDuration)` cho từng cặp `(giờ mở, thời lượng)`. `min(generator)` tìm giá trị nhỏ nhất mà không cần tạo danh sách. Hai vòng `for` lồng nhau trong generator duyệt **mọi cặp** `(i, j)`.

In [3]:
ls, ld = [2, 8], [4, 1]
print(list(zip(ls, ld)))
print(min(s + d for s, d in zip(ls, ld)))                       # thời điểm kết thúc sớm nhất nếu chơi ngay khi mở
print(min(a + b for a in [1, 5] for b in [10, 2]))

[(2, 4), (8, 1)]
6
3


### 3.3. Tìm kiếm nhị phân theo đáp án (ý tưởng nhanh)

Nếu có hàm `feasible(T)` ("có thể xong cả hai trước hoặc đúng lúc `T`?") và nó **đơn điệu** (đúng với `T` thì đúng với mọi `T' > T`), ta tìm `T` nhỏ nhất làm `feasible` đúng bằng nhị phân: `lo, hi`; `mid = (lo + hi) // 2`; nếu đúng thì `hi = mid`, sai thì `lo = mid + 1`.

In [4]:
def first_true(lo, hi, pred):
    while lo < hi:
        mid = (lo + hi) // 2
        if pred(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo

print(first_true(0, 100, lambda t: t >= 37))

37


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 2: `land=(5,3)`, `water=(1,10)`.

```
cạn trước : cạn 5 -> 8, nước vào lúc max(8,1)=8, xong 18
nước trước: nước 1 -> 11, cạn vào lúc max(11,5)=11, xong 14   <- tốt hơn
đáp án = min(18, 14) = 14
```

Hai thứ tự là **hai bài toán độc lập**; ta lấy `min`. Với mỗi thứ tự, du khách chọn một trò thứ nhất và một trò thứ hai. Câu hỏi: chọn thế nào cho kết thúc sớm nhất?

## 5. Cách 1 — Brute Force / baseline

Thử **tất cả** `(i, j)` và **cả hai** thứ tự:

- cạn `i` trước: `end1 = ls[i] + ld[i]`, `end = max(end1, ws[j]) + wd[j]`;
- nước `j` trước: `end1 = ws[j] + wd[j]`, `end = max(end1, ls[i]) + ld[i]`.

Lấy `min`. Nút thắt: `2 * n * m` tổ hợp (tối đa `20000`) — nhanh với đề, nhưng chưa tận dụng cấu trúc: trò thứ nhất tốt nhất không phụ thuộc trò thứ hai.

In [5]:
class SolutionBrute:
    def earliestFinishTime(self, landStartTime, landDuration, waterStartTime, waterDuration):
        best = float("inf")
        for i in range(len(landStartTime)):
            for j in range(len(waterStartTime)):
                # cạn trước
                end1 = landStartTime[i] + landDuration[i]
                best = min(best, max(end1, waterStartTime[j]) + waterDuration[j])
                # nước trước
                end1 = waterStartTime[j] + waterDuration[j]
                best = min(best, max(end1, landStartTime[i]) + landDuration[i])
        return best

print(SolutionBrute().earliestFinishTime([2, 8], [4, 1], [6], [3]))
print(SolutionBrute().earliestFinishTime([5], [3], [1], [10]))

9
14


**Độ phức tạp:** Time `O(n * m)`, Space `O(1)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** xét thứ tự "cạn trước, nước sau". Kết thúc cuối cùng là `max(f_i, ws_j) + wd_j` với `f_i = ls_i + ld_i`. Hàm này **không giảm theo `f_i`**: `f_i` càng nhỏ thì kết quả càng không tệ hơn. Do đó **chỉ cần trò cạn có `f_i` nhỏ nhất** (`F = min f_i`), không cần thử các trò cạn khác. Sau đó chọn trò nước `j` làm `max(F, ws_j) + wd_j` nhỏ nhất.

**Quy tắc "nếu… thì…":**

- Tính `F_land = min(ls + ld)` và `F_water = min(ws + wd)`.
- Đáp án cạn-trước: `min_j max(F_land, ws_j) + wd_j`.
- Đáp án nước-trước: `min_i max(F_water, ls_i) + ld_i`.
- Kết quả: `min` của hai thứ tự.

Lưu ý: ta chọn trò thứ nhất theo **thời điểm kết thúc** (không phải theo giờ mở hay thời lượng).

In [6]:
def greedy(ls, ld, ws, wd):
    F_land = min(s + d for s, d in zip(ls, ld))
    F_water = min(s + d for s, d in zip(ws, wd))
    land_first = min(max(F_land, s) + d for s, d in zip(ws, wd))
    water_first = min(max(F_water, s) + d for s, d in zip(ls, ld))
    print("F_land =", F_land, "F_water =", F_water, "| cạn trước:", land_first, "| nước trước:", water_first)
    return min(land_first, water_first)

print(greedy([2, 8], [4, 1], [6], [3]))
print(greedy([5], [3], [1], [10]))

F_land = 6 F_water = 9 | cạn trước: 9 | nước trước: 10
9
F_land = 8 F_water = 11 | cạn trước: 18 | nước trước: 14
14


## 7. Các cách tiếp theo

Tổng cộng 4 cách: (1) brute `O(nm)`; (2) tham lam `O(n + m)`; (3) tính sẵn thời điểm kết thúc rồi duyệt; (4) tìm kiếm nhị phân theo đáp án.

### Cách 2 — Tham lam `O(n + m)`

Giữ nguyên ý ở mục 6: hai lần duyệt để lấy `F_land`, `F_water`; hai lần duyệt nữa để chọn trò thứ hai.

In [7]:
class SolutionGreedy:
    def earliestFinishTime(self, landStartTime, landDuration, waterStartTime, waterDuration):
        f_land = min(s + d for s, d in zip(landStartTime, landDuration))
        f_water = min(s + d for s, d in zip(waterStartTime, waterDuration))
        land_first = min(max(f_land, s) + d for s, d in zip(waterStartTime, waterDuration))
        water_first = min(max(f_water, s) + d for s, d in zip(landStartTime, landDuration))
        return min(land_first, water_first)

print(SolutionGreedy().earliestFinishTime([2, 8], [4, 1], [6], [3]))
print(SolutionGreedy().earliestFinishTime([5], [3], [1], [10]))

9
14


Độ phức tạp Cách 2: Time `O(n + m)`, Space `O(1)`.

### Cách 3 — Hàm `finish_order` dùng chung + tính sẵn

Viết một hàm tổng quát `finish_order(A, B)` nhận hai nhóm trò (mỗi nhóm là danh sách `(start, dur)`), trả về thời điểm sớm nhất khi chơi một trò của `A` rồi một trò của `B`. Hàm này tính **một lần** danh sách thời điểm kết thúc của `A` rồi lấy min. Đáp án là `min(finish_order(land, water), finish_order(water, land))`. Cách này tránh lặp code của hai thứ tự.

In [8]:
class SolutionOrder:
    @staticmethod
    def finish_order(A, B):
        ends = [s + d for s, d in A]            # thời điểm kết thúc nếu chơi ngay khi mở
        first_done = min(ends)
        return min(max(first_done, s) + d for s, d in B)

    def earliestFinishTime(self, landStartTime, landDuration, waterStartTime, waterDuration):
        land = list(zip(landStartTime, landDuration))
        water = list(zip(waterStartTime, waterDuration))
        return min(self.finish_order(land, water), self.finish_order(water, land))

print(SolutionOrder().earliestFinishTime([2, 8], [4, 1], [6], [3]))
print(SolutionOrder().earliestFinishTime([5], [3], [1], [10]))

9
14


Độ phức tạp Cách 3: Time `O(n + m)`, Space `O(n + m)` cho danh sách cặp.

### Cách 4 — Tìm kiếm nhị phân theo đáp án

Đáp án nằm trong `[2, 3000]`: trò thứ nhất xong muộn nhất lúc `1000 + 1000 = 2000`, trò thứ hai kết thúc muộn nhất `max(2000, 1000) + 1000 = 3000`. `feasible(T)`: tồn tại cặp `(i, j)` và thứ tự sao cho kết thúc `<= T`. Hàm này đơn điệu theo `T`. Nhị phân tìm `T` nhỏ nhất đúng. Mỗi lần kiểm tra duyệt mọi cặp `O(nm)`, tổng `O(nm log 3000)`: chậm hơn, nhưng là khuôn mẫu "nhị phân theo đáp án" đáng nhớ.

In [9]:
class SolutionBinary:
    def earliestFinishTime(self, landStartTime, landDuration, waterStartTime, waterDuration):
        n, m = len(landStartTime), len(waterStartTime)

        def feasible(T):
            for i in range(n):
                for j in range(m):
                    if max(landStartTime[i] + landDuration[i], waterStartTime[j]) + waterDuration[j] <= T:
                        return True
                    if max(waterStartTime[j] + waterDuration[j], landStartTime[i]) + landDuration[i] <= T:
                        return True
            return False

        lo, hi = 2, 3000
        while lo < hi:
            mid = (lo + hi) // 2
            if feasible(mid):
                hi = mid
            else:
                lo = mid + 1
        return lo

print(SolutionBinary().earliestFinishTime([2, 8], [4, 1], [6], [3]))
print(SolutionBinary().earliestFinishTime([5], [3], [1], [10]))

9
14


Độ phức tạp Cách 4: Time `O(n * m * log 3000)`, Space `O(1)`.

## 8. TRACE từng bước

Trace Cách 2 với ví dụ 1 (`ls=[2,8]`, `ld=[4,1]`, `ws=[6]`, `wd=[3]`):

| bước | phép tính | giá trị |
|---|---|---|
| 1 | kết thúc từng trò cạn `ls+ld` | `[6, 9]`, `F_land = 6` |
| 2 | kết thúc từng trò nước `ws+wd` | `[9]`, `F_water = 9` |
| 3 | cạn trước: `max(6, 6) + 3` | `9` |
| 4 | nước trước: `max(9, 2) + 4`, `max(9, 8) + 1` | `13`, `10` -> min `10` |
| 5 | đáp án | `min(9, 10) = 9` |

In [10]:
def trace_rides(ls, ld, ws, wd):
    land_ends = [s + d for s, d in zip(ls, ld)]
    water_ends = [s + d for s, d in zip(ws, wd)]
    F_land, F_water = min(land_ends), min(water_ends)
    print("kết thúc cạn :", land_ends, "-> F_land =", F_land)
    print("kết thúc nước:", water_ends, "-> F_water =", F_water)
    cand_land_first = [(j, max(F_land, ws[j]) + wd[j]) for j in range(len(ws))]
    cand_water_first = [(i, max(F_water, ls[i]) + ld[i]) for i in range(len(ls))]
    print("cạn trước, chọn nước j ->", cand_land_first)
    print("nước trước, chọn cạn i ->", cand_water_first)
    ans = min(min(v for _, v in cand_land_first), min(v for _, v in cand_water_first))
    print("đáp án =", ans)
    return ans

trace_rides([2, 8], [4, 1], [6], [3])
trace_rides([5], [3], [1], [10])

kết thúc cạn : [6, 9] -> F_land = 6
kết thúc nước: [9] -> F_water = 9
cạn trước, chọn nước j -> [(0, 9)]
nước trước, chọn cạn i -> [(0, 13), (1, 10)]
đáp án = 9
kết thúc cạn : [8] -> F_land = 8
kết thúc nước: [11] -> F_water = 11
cạn trước, chọn nước j -> [(0, 18)]
nước trước, chọn cạn i -> [(0, 14)]
đáp án = 14


14

## 9. Code nộp LeetCode

In [11]:
from typing import List

class Solution:
    def earliestFinishTime(self, landStartTime: List[int], landDuration: List[int],
                           waterStartTime: List[int], waterDuration: List[int]) -> int:
        f_land = min(s + d for s, d in zip(landStartTime, landDuration))
        f_water = min(s + d for s, d in zip(waterStartTime, waterDuration))
        land_first = min(max(f_land, s) + d for s, d in zip(waterStartTime, waterDuration))
        water_first = min(max(f_water, s) + d for s, d in zip(landStartTime, landDuration))
        return min(land_first, water_first)

print(Solution().earliestFinishTime([2, 8], [4, 1], [6], [3]))
print(Solution().earliestFinishTime([5], [3], [1], [10]))

9
14


## 10. Vì sao thuật toán đúng

- **Hai thứ tự độc lập:** mọi phương án hợp lệ là (thứ tự, trò cạn `i`, trò nước `j`); đáp án là min trên cả hai thứ tự, nên xét riêng rồi lấy `min` là đủ.
- **Đơn điệu theo thời điểm xong trò thứ nhất:** với thứ tự cạn-trước, thời điểm cuối là `g(f) = max(f, ws_j) + wd_j`. `max(f, c)` không giảm theo `f`, nên `f` nhỏ hơn không bao giờ làm kết quả lớn hơn. Vì vậy với mọi `j`, trò cạn tối ưu là trò có `f` nhỏ nhất.
- **Trò thứ nhất bắt đầu ngay khi mở:** bắt đầu muộn hơn chỉ làm `f` lớn hơn, không có lợi (du khách không cần chờ trò thứ nhất).
- **Chọn trò thứ hai:** sau khi cố định `F`, ta thử mọi `j` và lấy min — đầy đủ.
- **Cách 4:** `feasible(T)` đơn điệu (nếu xong trước `T` thì cũng xong trước `T+1`), nên nhị phân ra `T` nhỏ nhất đạt được, bằng giá trị min của brute.

## 11. Phân tích Time / Space Complexity

`n` = số trò cạn, `m` = số trò nước.

| Cách | Time | Space |
|---|---|---|
| 1. Brute cặp + hai thứ tự | `O(n m)` | `O(1)` |
| 2. Tham lam | `O(n + m)` | `O(1)` |
| 3. `finish_order` dùng chung | `O(n + m)` | `O(n + m)` |
| 4. Nhị phân theo đáp án | `O(n m log V)`, `V = 3000` | `O(1)` |

Với `n, m <= 100` mọi cách đều rất nhanh; Cách 2 là tối ưu về thời gian và đơn giản nhất.

## 12. Bộ test

Test 1: ví dụ đề và edge case (giá trị trong `[1, 1000]`, `1 <= n, m <= 100`).

In [12]:
impls = [SolutionBrute(), SolutionGreedy(), SolutionOrder(), SolutionBinary(), Solution()]

def oracle(ls, ld, ws, wd):
    # mô phỏng độc lập từng cặp, từng thứ tự
    res = []
    for i in range(len(ls)):
        for j in range(len(ws)):
            t = ls[i]
            t += ld[i]
            if t < ws[j]:
                t = ws[j]
            res.append(t + wd[j])
            t = ws[j]
            t += wd[j]
            if t < ls[i]:
                t = ls[i]
            res.append(t + ld[i])
    return min(res)

tests = [
    (([2, 8], [4, 1], [6], [3]), 9),
    (([5], [3], [1], [10]), 14),
    (([1], [1], [1], [1]), 3),                      # 1->2, rồi 2->3
    (([1000], [1000], [1000], [1000]), 3000),
    (([1], [1], [10], [3]), 13),                    # nước mở muộn: chờ
    (([10, 1], [1, 8], [1], [1]), 10),              # trò kết thúc sớm nhất không phải trò thời lượng ngắn nhất
    (([1, 2], [10, 1], [5], [2]), 7),               # trò mở sớm nhất không phải trò kết thúc sớm nhất
    (([3, 3], [2, 2], [3, 3], [2, 2]), 7),          # hai trò giống hệt nhau
    (([1] * 100, [1000] * 100, [1000] * 100, [1] * 100), 1002),
]
for args, exp in tests:
    ls, ld, ws, wd = args
    assert 1 <= len(ls) == len(ld) <= 100 and 1 <= len(ws) == len(wd) <= 100
    assert all(1 <= v <= 1000 for v in ls + ld + ws + wd)
    assert oracle(*args) == exp, (args, oracle(*args), exp)
    for im in impls:
        assert im.earliestFinishTime(*[list(a) for a in args]) == exp, (type(im).__name__, args)
print("tất cả test cố định OK:", len(tests))

tất cả test cố định OK: 9


Test 2: ngẫu nhiên có seed. Sinh nhiều kiểu: khoảng giá trị nhỏ (nhiều bằng nhau), khoảng đầy đủ `[1,1000]`, và kích thước từ `1` đến `12` (Cách 4 chậm hơn nên giới hạn cho vòng này).

In [13]:
import random
random.seed(3633)

for t in range(2500):
    n, m = random.randint(1, 12), random.randint(1, 12)
    hi = random.choice([3, 10, 50, 1000])
    ls = [random.randint(1, hi) for _ in range(n)]
    ld = [random.randint(1, hi) for _ in range(n)]
    ws = [random.randint(1, hi) for _ in range(m)]
    wd = [random.randint(1, hi) for _ in range(m)]
    exp = oracle(ls, ld, ws, wd)
    for im in impls:
        assert im.earliestFinishTime(list(ls), list(ld), list(ws), list(wd)) == exp, (type(im).__name__, ls, ld, ws, wd)
print("2500 test ngẫu nhiên OK")

2500 test ngẫu nhiên OK


Test 3: tính chất — đáp án **đối xứng** khi đổi vai trò cạn/nước; đáp án `>= 2` và `<= 3000`; thêm một trò mới **không bao giờ làm đáp án lớn hơn**; dịch tất cả giờ mở thêm `c` không làm đáp án tăng quá `c`. Dùng kích thước tối đa `100` cho Cách 2 (nhanh).

In [14]:
random.seed(633)
g = SolutionGreedy()
for _ in range(500):
    n, m = random.randint(1, 100), random.randint(1, 100)
    ls = [random.randint(1, 1000) for _ in range(n)]
    ld = [random.randint(1, 1000) for _ in range(n)]
    ws = [random.randint(1, 1000) for _ in range(m)]
    wd = [random.randint(1, 1000) for _ in range(m)]
    base = g.earliestFinishTime(ls, ld, ws, wd)
    assert base == g.earliestFinishTime(ws, wd, ls, ld)          # đối xứng
    assert 2 <= base <= 3000
    if n < 100:
        more = g.earliestFinishTime(ls + [random.randint(1, 1000)], ld + [random.randint(1, 1000)], ws, wd)
        assert more <= base                                      # thêm trò không làm tệ đi
print("tính chất (đối xứng, biên, thêm trò không tệ hơn): OK")
for _ in range(50):
    n, m = random.randint(1, 100), random.randint(1, 100)
    ls = [random.randint(1, 1000) for _ in range(n)]; ld = [random.randint(1, 1000) for _ in range(n)]
    ws = [random.randint(1, 1000) for _ in range(m)]; wd = [random.randint(1, 1000) for _ in range(m)]
    assert g.earliestFinishTime(ls, ld, ws, wd) == SolutionBrute().earliestFinishTime(ls, ld, ws, wd)
print("kích thước tối đa: Greedy == Brute OK")

tính chất (đối xứng, biên, thêm trò không tệ hơn): OK
kích thước tối đa: Greedy == Brute OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: chỉ xét một thứ tự (cạn trước)

In [15]:
def wrong_one_order(ls, ld, ws, wd):
    F = min(s + d for s, d in zip(ls, ld))
    return min(max(F, s) + d for s, d in zip(ws, wd))       # SAI: bỏ thứ tự nước-trước

args = ([5], [3], [1], [10])
print("sai :", wrong_one_order(*args))
print("đúng:", Solution().earliestFinishTime(*args))
assert wrong_one_order(*args) != Solution().earliestFinishTime(*args)

sai : 18
đúng: 14


Vì sao sai: ví dụ 2 cần nước trước (`14`), còn cạn trước cho `18`. **Sửa:** tính cả hai thứ tự rồi lấy `min`.

### Lỗi 2: quên chờ giờ mở cửa của trò thứ hai

In [16]:
def wrong_no_wait(ls, ld, ws, wd):
    F_land = min(s + d for s, d in zip(ls, ld))
    F_water = min(s + d for s, d in zip(ws, wd))
    a = F_land + min(wd)               # SAI: bỏ qua giờ mở cửa của trò nước
    b = F_water + min(ld)
    return min(a, b)

args = ([1], [1], [10], [3])
print("sai :", wrong_no_wait(*args))
print("đúng:", Solution().earliestFinishTime(*args))
assert wrong_no_wait(*args) != Solution().earliestFinishTime(*args)

sai : 5
đúng: 13


Vì sao sai: cạn xong lúc `2` nhưng nước chỉ mở lúc `10`, nên nước xong lúc `13`, không phải `5`. **Sửa:** `max(f, giờ mở) + duration`.

### Lỗi 3: chọn trò thứ nhất theo giờ **mở sớm nhất**

In [17]:
def wrong_earliest_start(ls, ld, ws, wd):
    i = ls.index(min(ls)); j = ws.index(min(ws))               # SAI: chọn theo giờ mở
    a = max(ls[i] + ld[i], ws[j]) + wd[j]
    b = max(ws[j] + wd[j], ls[i]) + ld[i]
    return min(a, b)

args = ([1, 2], [10, 1], [5], [2])
print("sai :", wrong_earliest_start(*args))
print("đúng:", Solution().earliestFinishTime(*args))
assert wrong_earliest_start(*args) != Solution().earliestFinishTime(*args)

sai : 13
đúng: 7


Vì sao sai: trò cạn 0 mở lúc `1` nhưng kéo dài `10` (xong `11`); trò cạn 1 mở lúc `2`, chỉ `1` (xong `3`). Đáp án đúng `7`. **Sửa:** chọn theo **thời điểm kết thúc** `start + duration`.

### Lỗi 4: chọn trò thứ nhất theo thời lượng **ngắn nhất**

In [18]:
def wrong_shortest_duration(ls, ld, ws, wd):
    i = ld.index(min(ld)); j = wd.index(min(wd))                # SAI: chọn theo thời lượng
    a = max(ls[i] + ld[i], ws[j]) + wd[j]
    b = max(ws[j] + wd[j], ls[i]) + ld[i]
    return min(a, b)

args = ([10, 1], [1, 8], [1], [1])
print("sai :", wrong_shortest_duration(*args))
print("đúng:", Solution().earliestFinishTime(*args))
assert wrong_shortest_duration(*args) != Solution().earliestFinishTime(*args)

sai : 11
đúng: 10


Vì sao sai: trò cạn 0 ngắn nhất (`1`) nhưng mở muộn (`10`, xong `11`); trò cạn 1 dài `8` nhưng mở sớm (xong `9`). Đáp án đúng `10`, tính sai ra `11`. **Sửa:** so sánh `start + duration`, không phải `duration`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Hai nhóm, chọn một mỗi nhóm, thứ tự bất kỳ" -> thử **cả hai thứ tự**, mỗi thứ tự giải riêng.
- Công thức hay gặp: `bắt đầu = max(vừa xong, giờ mở)`.
- Khi hàm mục tiêu **không giảm** theo biến của bước trước -> chọn bước trước **tốt nhất cục bộ** (tham lam).
- Với `n, m <= 100`, brute `O(nm)` cũng đạt; nhưng luôn tự hỏi "có tối ưu độc lập không?".
- Mẫu "nhị phân theo đáp án" khi có hàm kiểm tra đơn điệu.
- Sinh test nhỏ + oracle brute là cách chắc chắn nhất để bắt lỗi tham lam.

## 15. Biến thể của bài

1. **Phần II** của bài (ride II): du khách có thể chọn thời điểm bắt đầu trò thứ nhất khác — thay đổi cách tính, vẫn tham lam.
2. Phải chơi **cả hai loại theo thứ tự cố định** (chỉ cạn rồi nước).
3. Chơi `k` trò thay vì hai (cần quy hoạch động hoặc tham lam khác).
4. Hỏi **cặp trò** (chỉ số `i, j`) đạt đáp án, hoặc đếm số cặp tối ưu.
5. Thêm **thời gian đi lại** cố định giữa hai trò.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute cặp + hai thứ tự | thử tất cả | `O(nm)` | `O(1)` | oracle, kiểm chứng |
| 2. Tham lam | trò thứ nhất kết thúc sớm nhất | `O(n+m)` | `O(1)` | **nên nộp** |
| 3. `finish_order` dùng chung | tách hàm theo thứ tự | `O(n+m)` | `O(n+m)` | code sạch, tránh lặp |
| 4. Nhị phân theo đáp án | `feasible(T)` đơn điệu | `O(nm log V)` | `O(1)` | luyện mẫu nhị phân theo đáp án |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().earliestFinishTime([3], [4], [3], [4])` là gì?

**Bài 2 (tính tay):** `ls=[1,6]`, `ld=[2,1]`, `ws=[4]`, `wd=[2]`. Tính `F_land`, hai thứ tự và đáp án.

**Bài 3 (điền TODO):** hoàn thành `one_order(A, B)` trả về thời điểm sớm nhất chơi một trò của `A` rồi một trò của `B` (`A`, `B` là danh sách `(start, dur)`).

**Bài 4 (tự viết lại):** viết `best_pair(ls, ld, ws, wd)` trả về **thứ tự và cặp chỉ số** `(order, i, j)` đạt đáp án (`order` là `"land"` hoặc `"water"` nghĩa là chơi trò đó trước).

**Bài 5 (mini challenge):** nếu giữa hai trò phải **đi bộ `w` đơn vị thời gian**, đáp án thay đổi thế nào? Viết `earliest_with_walk(ls, ld, ws, wd, w)`.

In [19]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().earliestFinishTime([3], [4], [3], [4]))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def one_order(A, B):
    # TODO: first_done = min(s + d for s, d in A); trả min(max(first_done, s) + d for s, d in B)
    return -1

11


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** với mỗi thứ tự, tìm thời điểm kết thúc sớm nhất của trò thứ nhất, rồi dùng công thức `max(f, giờ mở) + duration` cho trò thứ hai.

**Mức 2 (vừa):** Bài 3: hai dòng `min(...)`. Bài 4: duyệt mọi `(i, j)` và hai thứ tự, giữ `(giá trị, order, i, j)` nhỏ nhất. Bài 5: kết thúc trò thứ nhất cộng thêm `w` trước khi `max` với giờ mở.

**Mức 3 (gần đáp án):** Bài 1: `3 -> 7`, rồi `max(7, 3) + 4 = 11`; hai thứ tự đối xứng -> `11`. Bài 2: `F_land = min(3, 7) = 3`, cạn trước: `max(3,4)+2 = 6`; `F_water = 6`, nước trước: `max(6,1)+2 = 8` hoặc `max(6,6)+1 = 7`; đáp án `6`.

## 19. Đáp án tham khảo cho bài tập

In [20]:
# Bài 1
assert Solution().earliestFinishTime([3], [4], [3], [4]) == 11

# Bài 2
assert Solution().earliestFinishTime([1, 6], [2, 1], [4], [2]) == 6

# Bài 3
def one_order(A, B):
    first_done = min(s + d for s, d in A)
    return min(max(first_done, s) + d for s, d in B)
assert one_order([(2, 4), (8, 1)], [(6, 3)]) == 9
assert one_order([(6, 3)], [(2, 4), (8, 1)]) == 10

# Bài 4
def best_pair(ls, ld, ws, wd):
    best = None
    for i in range(len(ls)):
        for j in range(len(ws)):
            for order in ("land", "water"):
                if order == "land":
                    val = max(ls[i] + ld[i], ws[j]) + wd[j]
                else:
                    val = max(ws[j] + wd[j], ls[i]) + ld[i]
                if best is None or val < best[0]:
                    best = (val, order, i, j)
    return best[1], best[2], best[3]
assert best_pair([5], [3], [1], [10]) == ("water", 0, 0)
o, i, j = best_pair([2, 8], [4, 1], [6], [3])
assert (o, i, j) == ("land", 0, 0)

# Bài 5
def earliest_with_walk(ls, ld, ws, wd, w):
    f_land = min(s + d for s, d in zip(ls, ld))
    f_water = min(s + d for s, d in zip(ws, wd))
    a = min(max(f_land + w, s) + d for s, d in zip(ws, wd))
    b = min(max(f_water + w, s) + d for s, d in zip(ls, ld))
    return min(a, b)
assert earliest_with_walk([2, 8], [4, 1], [6], [3], 0) == Solution().earliestFinishTime([2, 8], [4, 1], [6], [3])
assert earliest_with_walk([2, 8], [4, 1], [6], [3], 5) == 14
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "hai nhóm, một thứ tự cố định, thời điểm xong nhỏ nhất":**

```python
def one_order(A, B):                    # A: danh sách (start, dur) chơi trước
    first_done = min(s + d for s, d in A)
    return min(max(first_done, s) + d for s, d in B)

return min(one_order(land, water), one_order(water, land))
```

**Checklist trước khi submit:**

- [ ] Đã xét **cả hai thứ tự** (cạn trước, nước trước)?
- [ ] Trò thứ hai dùng `max(vừa xong, giờ mở)` (chờ nếu đến sớm)?
- [ ] Trò thứ nhất chọn theo **thời điểm kết thúc** `start + duration` nhỏ nhất, không theo giờ mở hay thời lượng?
- [ ] Đúng **một** trò mỗi loại (không chơi nhiều hơn)?
- [ ] Đã thử `n = m = 1` và các ví dụ đề (`9` và `14`)?
- [ ] Chữ ký đúng `earliestFinishTime(self, landStartTime, landDuration, waterStartTime, waterDuration) -> int`?